# M7 · v0.5 · Equipo de agentes con aprobación humana

Tu asistente como **equipo** (supervisor, investigador, analista, redactor) y **una persona que aprueba**. Medís si vale lo que cuesta frente a v0.4. Es tu **v0.5**.

**La tarea, en cuatro pasos:** 1. **Corré** el grafo y dibujalo en mermaid.live · 2. **Aprobá y rechazá** un borrador · 3. **Predecí** dónde gana el equipo (en `PREDICCION`) · 4. **Compará** v0.4 contra v0.5 y **decidí**.

- Todo corre acá, en Colab: no hay archivos ni carpetas.
- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
- **`buscar_web` da ERROR** (`RatelimitException`, `TimeoutException`): DuckDuckGo cortó. Para medir, `WEB = "grabada"`.
</details>


## 1 · Preparar (una vez, menos de un minuto)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2 chromadb==1.5.9 langchain-text-splitters==1.1.3 ddgs==9.16.0
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

import time

import pandas as pd
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.5-flash-lite"
print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

## 2 · Tus documentos

Tocá **Elegir archivos** y subí tus `.md` o `.txt` (al empezar, los 5 de ejemplo del aula virtual).
Un PDF: copiá el texto a un `.txt`. Si reiniciás Colab, se suben de nuevo.

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    subidos = files.upload()
except ImportError:  # fuera de Colab (solo para probar el cuaderno): una carpeta local
    carpeta = Path(os.environ.get("DOCUMENTOS", "corpus"))
    subidos = {p.name: p.read_bytes() for p in carpeta.glob("*") if p.suffix.lower() in (".md", ".txt") and not p.name.startswith("LICENSE")}
documentos = {Path(n).stem: b.decode("utf-8") for n, b in subidos.items() if n.lower().endswith((".md", ".txt"))}
print(len(documentos), "documentos:", ", ".join(documentos))

## 3 · Índice, herramientas y tu v0.4

Lo mismo que en M6 (copiá tus descripciones y tu `SISTEMA`): v0.4 es la vara para comparar.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK = 1000
OVERLAP = 150
K = 3


def indexar(chunk, overlap):
    """Parte cada documento en fragmentos y los guarda en Chroma (en memoria): busca por significado."""
    nombre = f"docs_{chunk}_{overlap}_{len(documentos)}"
    coleccion = chromadb.Client().get_or_create_collection(nombre, embedding_function=DefaultEmbeddingFunction())
    if coleccion.count():
        return coleccion
    cortador = RecursiveCharacterTextSplitter(chunk_size=chunk, chunk_overlap=overlap)
    for doc, texto in documentos.items():
        partes = cortador.split_text(texto)
        coleccion.add(ids=[f"{doc}#{i}" for i in range(len(partes))], documents=partes,
                      metadatas=[{"doc": doc, "chunk": i} for i in range(len(partes))])
    return coleccion


def buscar(coleccion, pregunta, k):
    """Los k fragmentos más parecidos a la pregunta (distancia más chica = más parecido)."""
    r = coleccion.query(query_texts=[pregunta], n_results=k)
    return [{"doc": m["doc"], "chunk": m["chunk"], "distancia": round(d, 3), "texto": t}
            for m, d, t in zip(r["metadatas"][0], r["distances"][0], r["documents"][0])]


coleccion = indexar(CHUNK, OVERLAP)
print(coleccion.count(), "fragmentos")

In [ ]:
import re
from typing import Literal

from langchain.tools import tool

WEB = "grabada"   # "real" (busca de verdad), "grabada" (siempre lo mismo: mide la elección) o "caida" (simula la caída)
RECORTE = 600     # caracteres de cada fragmento que ve el agente
GRABADA = ("- langgraph · PyPI: LangGraph is a low-level orchestration framework for building stateful agents\n"
           "- chromadb · PyPI: Chroma, the AI-native open-source embedding database\n"
           "- LangGraph es desarrollado por LangChain Inc. (github.com/langchain-ai/langgraph)")


@tool
def buscar_documentos(consulta: str) -> str:
    """Busca en la documentación del corpus (LangGraph/LangChain: persistencia, interrupts,
    subgrafos, memoria). Usar para preguntas técnicas sobre esos temas."""
    return "\n\n".join(f"[{h['doc']}] {h['texto'][:RECORTE]}" for h in buscar(coleccion, consulta, K))


@tool
def buscar_web(consulta: str) -> str:
    """Busca en internet información reciente o que no está en la documentación del corpus
    (noticias, precios, versiones nuevas, empresas, fechas)."""
    if WEB == "caida":
        return "ERROR: la búsqueda web no respondió. No inventes; decíselo al usuario."
    if WEB == "grabada":
        return GRABADA
    try:
        from ddgs import DDGS  # DuckDuckGo: gratis y sin clave
        return "\n".join(f"- {r['title']}: {r['body'][:300]}" for r in DDGS().text(consulta, max_results=3)) or "Sin resultados."
    except Exception as error:
        return f"ERROR: la búsqueda web falló ({type(error).__name__}). No inventes; decíselo al usuario."


@tool
def calcular(operacion: Literal["suma", "resta", "multiplicacion", "division", "porcentaje_cambio"],
             a: float, b: float) -> str:
    """Hace una cuenta con dos números. porcentaje_cambio = (b - a) / a * 100. Usar para cualquier cuenta."""
    resultado = {"suma": a + b, "resta": a - b, "multiplicacion": a * b, "division": a / b if b else None,
                 "porcentaje_cambio": (b - a) / a * 100 if a else None}[operacion]
    return "ERROR: división por cero." if resultado is None else f"{round(resultado, 4) + 0.0:.12g}"


def fuentes_de(mensajes):
    """Los documentos que trajo buscar_documentos (para el hit)."""
    docs = [d for m in mensajes if getattr(m, "name", None) == "buscar_documentos"
            for d in re.findall(r"^\[([^\]]+)\]", str(m.content), re.M)]
    return list(dict.fromkeys(docs))

In [ ]:
import uuid

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver

SISTEMA = ("Sos un asistente de investigación. Usá buscar_documentos para LangGraph/LangChain, "
           "buscar_web para información reciente o externa y calcular para cualquier cuenta. "
           "Si una herramienta devuelve ERROR, decilo y no inventes. Respondé en español, breve.")
MAX_LLAMADAS_MODELO = 6
MAX_LLAMADAS_HERRAMIENTAS = 5


def topes():  # los límites van en código, no en el prompt
    return [ModelCallLimitMiddleware(run_limit=MAX_LLAMADAS_MODELO),
            ToolCallLimitMiddleware(run_limit=MAX_LLAMADAS_HERRAMIENTAS)]


modelo = init_chat_model(MODELO)
agente = create_agent(modelo, tools=[buscar_documentos, buscar_web, calcular], system_prompt=SISTEMA,
                      middleware=topes(), checkpointer=InMemorySaver())


class Contador(BaseCallbackHandler):
    """Cuenta las llamadas al modelo (cada una cuesta)."""
    def __init__(self):
        self.llamadas = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.llamadas += 1


def texto_final(mensajes):
    texto = mensajes[-1].text
    return "No llegué a una respuesta: se alcanzó el tope de llamadas." if texto.startswith("Model call limits") else texto


def responder_v04(pregunta):
    contador = Contador()
    mensajes = agente.invoke({"messages": [{"role": "user", "content": pregunta}]},
                             {"configurable": {"thread_id": uuid.uuid4().hex}, "callbacks": [contador]})["messages"]
    return {"respuesta": texto_final(mensajes), "herramientas": [p["name"] for m in mensajes for p in getattr(m, "tool_calls", [])],
            "llamadas": contador.llamadas, "fuentes": fuentes_de(mensajes)}

## 4 · El grafo: supervisor, especialistas y la firma

Perillas: `MAX_PASOS`, `INVESTIGADOR`, `ANALISTA` y `SUPERVISOR`. La celda imprime el dibujo: copialo (del primer `---` a la última `classDef`) en [mermaid.live](https://mermaid.live).

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

MAX_PASOS = 3
INVESTIGADOR = ("Investigás: buscar_documentos para LangGraph/LangChain, buscar_web para datos externos "
                "o recientes. Devolvé hallazgos breves con su fuente. Si una herramienta da ERROR, decilo.")
ANALISTA = "Hacés cuentas con calcular sobre los hallazgos. Breve."
SUPERVISOR = "Elegí el próximo paso. investigador: falta información. analista: hay que calcular. redactor: ya alcanza."

investigador = create_agent(modelo, tools=[buscar_documentos, buscar_web], system_prompt=INVESTIGADOR, middleware=topes())
analista = create_agent(modelo, tools=[calcular], system_prompt=ANALISTA, middleware=topes())


class Ruta(BaseModel):
    siguiente: Literal["investigador", "analista", "redactor"]


class Estado(TypedDict, total=False):
    pregunta: str
    notas: Annotated[list[str], operator.add]          # cada especialista SUMA notas
    herramientas: Annotated[list[str], operator.add]
    fuentes: Annotated[list[str], operator.add]
    pasos: int
    siguiente: str
    borrador: str
    aprobado: bool
    respuesta_final: str


def supervisor(e):
    if e.get("pasos", 0) >= MAX_PASOS:  # tope en código
        return {"siguiente": "redactor"}
    try:
        ruta = modelo.with_structured_output(Ruta).invoke(f"{SUPERVISOR}\nPregunta: {e['pregunta']}\nNotas: {e.get('notas') or 'ninguna'}")
    except ValueError:
        ruta = None
    return {"siguiente": ruta.siguiente if ruta else "redactor", "pasos": e.get("pasos", 0) + 1}


def especialista(agente_nodo):
    def nodo(e):
        mensajes = agente_nodo.invoke({"messages": [{"role": "user", "content": f"{e['pregunta']}\nNotas previas: {e.get('notas') or 'ninguna'}"}]})["messages"]
        return {"notas": [texto_final(mensajes)], "herramientas": [p["name"] for m in mensajes for p in getattr(m, "tool_calls", [])],
                "fuentes": fuentes_de(mensajes)}
    return nodo


def redactor(e):
    return {"borrador": modelo.invoke(f"Respondé '{e['pregunta']}' en 2 o 3 frases, SOLO con estas notas; si no alcanzan, decilo.\n"
                                      f"Notas: {e.get('notas') or 'ninguna'}").text}


def aprobar(e):
    decision = interrupt({"borrador": e["borrador"]})  # PAUSA: el estado queda guardado
    return {"aprobado": bool(decision.get("aprobado", False)) if isinstance(decision, dict) else decision is True}


g = StateGraph(Estado)
for nombre, funcion in [("supervisor", supervisor), ("investigador", especialista(investigador)), ("analista", especialista(analista)),
                        ("redactor", redactor), ("aprobar", aprobar), ("final", lambda e: {"respuesta_final": e["borrador"]})]:
    g.add_node(nombre, funcion)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda e: e["siguiente"], {"investigador": "investigador", "analista": "analista", "redactor": "redactor"})
g.add_edge("investigador", "supervisor")
g.add_edge("analista", "supervisor")
g.add_edge("redactor", "aprobar")
g.add_conditional_edges("aprobar", lambda e: "final" if e["aprobado"] else END, {"final": "final", END: END})
g.add_edge("final", END)
grafo = g.compile(checkpointer=InMemorySaver())


def correr(pregunta, hilo, aprobado, contador=None):
    config = {"configurable": {"thread_id": hilo}, "callbacks": [contador] if contador else []}
    pausa = grafo.invoke({"pregunta": pregunta}, config)
    final = grafo.invoke(Command(resume={"aprobado": aprobado}), config)  # mismo hilo = mismo estado
    return pausa, final


def responder_v05(pregunta):  # para la suite: la persona aprueba siempre (medimos calidad)
    contador = Contador()
    _, final = correr(pregunta, uuid.uuid4().hex, True, contador)
    return {"respuesta": final.get("respuesta_final", ""), "herramientas": final.get("herramientas", []),
            "fuentes": list(dict.fromkeys(final.get("fuentes", []))), "llamadas": contador.llamadas}


print(grafo.get_graph().draw_mermaid())

## 5 · Aprobar y rechazar

Cambiá la pregunta por una de tu tema que necesite investigar **y** calcular.

In [ ]:
pregunta = "Según la documentación, ¿cuántas filas tiene la tabla que compara checkpointer y store? Multiplicá por 3."
pausa, final = correr(pregunta, "demo-aprueba", True)
print("Borrador:", pausa.get("borrador"))
print("Notas:", pausa.get("notas"))
print("Con aprobación → respuesta final:", final.get("respuesta_final"))
_, final = correr(pregunta, "demo-rechaza", False)
print("Con rechazo → respuesta final:", final.get("respuesta_final", "(ninguna: el rechazo la bloqueó)"))
print(grafo.get_state({"configurable": {"thread_id": "demo-aprueba"}}).values.keys())

**Qué deberías ver:** el borrador, las notas de cada especialista, la respuesta con aprobación y **ninguna** con rechazo; abajo, las claves del estado guardado.

**Primer control · 10–15 min:** el dibujo en mermaid.live, el borrador y las dos respuestas.

## 6 · ¿El equipo vale lo que cuesta?

In [ ]:
SUITE = [  # reemplazala por la tuya: unas 10, al menos una de cada tipo
    {"id": 1, "tipo": "corpus", "pregunta": "¿Qué hace interrupt() en LangGraph?", "esperado": "Pausa el grafo y espera una entrada externa.", "herramientas": ["buscar_documentos"]},
    {"id": 2, "tipo": "web", "pregunta": "¿Qué empresa desarrolla LangGraph?", "esperado": "LangChain.", "herramientas": ["buscar_web"]},
    {"id": 3, "tipo": "calculo", "pregunta": "Si una consulta cuesta 0.0042 USD, ¿cuánto cuestan 1500 consultas?", "esperado": "6.3 USD.", "herramientas": ["calcular"]},
    # {"id": 4, "tipo": "ninguna", "pregunta": "Hola, ¿cómo estás?", "esperado": "Saludo, sin herramientas.", "herramientas": []},
    {"id": 5, "tipo": "combinada", "pregunta": "Según la documentación, ¿cuántas filas tiene la tabla que compara checkpointer y store? Multiplicá ese número por 3.", "esperado": "15.", "herramientas": ["buscar_documentos", "calcular"]},
]

In [ ]:
def correr_suite(responder, suite, tag):
    """Corre la suite, imprime el resumen y devuelve una tabla con una fila por pregunta."""
    filas = []
    for caso in suite:
        inicio = time.perf_counter()
        try:
            salida = responder(caso["pregunta"])
        except Exception as error:  # cuota, red o proveedor saturado: esa fila no mide tu sistema
            salida = {"respuesta": f"ERROR: {type(error).__name__}"}
        fila = {"id": caso["id"], "tipo": caso["tipo"], "pregunta": caso["pregunta"], "esperado": caso["esperado"],
                "respuesta": salida["respuesta"], "fuentes": salida.get("fuentes", []),
                "herramientas": salida.get("herramientas", []), "llamadas": salida.get("llamadas"),
                "latencia_s": round(time.perf_counter() - inicio, 2)}
        if "fuentes" in caso:  # recuperación y abstención
            abstuvo = "NO ENCONTRADO" in fila["respuesta"].upper()
            fila["hit"] = None if not caso["fuentes"] else any(f in fila["fuentes"] for f in caso["fuentes"])
            fila["abstencion_ok"] = abstuvo == (not caso["fuentes"])
        if "herramientas" in caso:  # selección de herramientas
            usadas, esperadas = set(fila["herramientas"]), set(caso["herramientas"])
            fila["seleccion_ok"] = usadas == esperadas if not esperadas else esperadas <= usadas
        filas.append(fila)
    tabla = pd.DataFrame(filas)
    partes = []
    if "hit" in tabla:
        con_fuente = tabla["hit"].dropna()
        partes += [f"hit {int(con_fuente.sum())}/{len(con_fuente)}",
                   f"abstención correcta {int(tabla['abstencion_ok'].sum())}/{len(tabla)}"]
    if "seleccion_ok" in tabla:
        partes.append(f"selección {int(tabla['seleccion_ok'].sum())}/{len(tabla)}")
    if tabla["llamadas"].notna().any():
        partes.append(f"llamadas/consulta {tabla['llamadas'].mean():.1f}")
    partes.append(f"latencia mediana {tabla['latencia_s'].median():.1f}s")
    print(f"{tag}: " + " · ".join(partes))
    errores = tabla[tabla["respuesta"].str.startswith("ERROR: ")]
    if len(errores):  # no miden tu sistema: 429 (cuota), 503 (saturado) o red
        print(f"Ojo: {len(errores)} preguntas dieron ERROR:", dict(zip(errores["id"], errores["respuesta"])), "→ corré de nuevo más tarde.")
    return tabla

In [ ]:
PREDICCION = ""  # antes de correr: ¿dónde gana el equipo? ¿Cuántas llamadas más? Por ejemplo: "en las de calcular; el doble"


In [ ]:
print("Tu predicción:", PREDICCION or "(todavía no la escribiste: hacelo antes de correr)")
tabla_v04 = correr_suite(responder_v04, SUITE, "v04")
tabla_v05 = correr_suite(responder_v05, SUITE, "v05")

**Qué deberías ver:** una línea por versión, con `selección`, `llamadas/consulta` y `latencia mediana`. Medido el 21/10 con estas 4 preguntas: v0.4, 2,2 llamadas por consulta; v0.5, 5,2 y 6,7 (de 2,4 a 3 veces más). Y una corrida de la demo contó mal la tabla: el redactor no comprueba.

## Entregar

- **Guardá este notebook en tu repo:** *Archivo → Guardar una copia en GitHub* → tu repositorio → `v05.ipynb`.
- **Evidencia** (una celda de texto al final, acá mismo): el dibujo del grafo, aprobar y rechazar, la tabla v0.4 vs v0.5 y **tu decisión** (si gana el agente solo, también vale).

**Microdefensa:** ¿Qué pasa del investigador al redactor, y por qué no toda la conversación? Confianza del 1 al 4.